In [ ]:
# @title
!pip install longport

In [ ]:
# @title
# ============================================================
# ULTIMATE STRATEGY TERMINAL - FINAL STABLE VERSION
# ============================================================

import datetime as dt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import longport.openapi as lp

# ---------- 环境 ----------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pass

ctx = None

# ============================================================
# 工具函数
# ============================================================

def safe_float(x, default=0.0):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default

def format_wan(v):
    try:
        return f"{v/10000:.2f}万"
    except Exception:
        return "--"

def interp_flip(df_net):
    if df_net is None or df_net.empty:
        return None
    df_net = df_net.sort_values("Strike").reset_index(drop=True)
    xs, ys = df_net["Strike"].values, df_net["NetGEX"].values
    for i in range(len(df_net) - 1):
        if ys[i] * ys[i + 1] < 0:
            return float(xs[i] + (-ys[i]) * (xs[i + 1] - xs[i]) / (ys[i + 1] - ys[i]))
    return None

def calculate_max_pain(df):
    if df is None or df.empty:
        return None, [], []
    strikes = sorted(df["Strike"].unique())
    pains = []
    for s in strikes:
        calls = df[(df["Type"] == "C") & (df["Strike"] < s)]
        c_sum = ((s - calls["Strike"]) * calls["OI"]).sum()
        puts = df[(df["Type"] == "P") & (df["Strike"] > s)]
        p_sum = ((puts["Strike"] - s) * puts["OI"]).sum()
        pains.append(c_sum + p_sum)
    max_pain_strike = strikes[np.argmin(pains)]
    return max_pain_strike, strikes, pains

def clear_outputs(*outs):
    for out in outs:
        with out:
            clear_output(wait=True)

def create_pagination_widget(df, rows_per_page=10):
    if df is None or df.empty:
        return widgets.HTML("<div style='color:#F87171;padding:10px;'>⚠️ 无数据</div>")

    df_raw = df.copy()

    page_input = widgets.BoundedIntText(
        value=1, min=1, max=100, description='页码:',
        layout={'width': '95px'}
    )
    prev_btn = widgets.Button(description="上一页", layout={'width': '78px'})
    next_btn = widgets.Button(description="下一页", layout={'width': '78px'})
    sort_dropdown = widgets.Dropdown(
        options=['原始顺序'] + list(df.columns),
        value='原始顺序',
        description='排序:',
        layout={'width': '220px'}
    )
    sort_order = widgets.Dropdown(
        options=[('降序 ⬇️', False), ('升序 ⬆️', True)],
        value=False,
        description='方式:',
        layout={'width': '135px'}
    )
    output_box = widgets.Output(layout={'width': '100%'})

    def show_page(page):
        if sort_dropdown.value == '原始顺序':
            view = df_raw
        else:
            view = df_raw.sort_values(by=sort_dropdown.value, ascending=sort_order.value)

        total_pages = max(1, (len(view) - 1) // rows_per_page + 1)
        page_input.max = total_pages
        page = min(max(1, page), total_pages)

        with output_box:
            clear_output(wait=True)
            start = (page - 1) * rows_per_page
            end = page * rows_per_page
            display(view.iloc[start:end])

    def on_ui_change(change):
        show_page(page_input.value)

    prev_btn.on_click(lambda _: setattr(page_input, 'value', max(1, page_input.value - 1)))
    next_btn.on_click(lambda _: setattr(page_input, 'value', min(page_input.max, page_input.value + 1)))

    for w in [page_input, sort_dropdown, sort_order]:
        w.observe(on_ui_change, names='value')

    show_page(1)

    toolbar = widgets.HBox(
        [prev_btn, next_btn, page_input, sort_dropdown, sort_order],
        layout=widgets.Layout(
            flex_flow='row wrap',
            align_items='center',
            gap='6px',
            margin='0 0 8px 0',
            width='100%'
        )
    )

    return widgets.VBox(
        [toolbar, output_box],
        layout=widgets.Layout(width='100%', overflow='hidden')
    )

# ============================================================
# UI 组件
# ============================================================

def card_box(child, title="", width="100%", height=None, padding="10px",
             header_bg="linear-gradient(90deg,#07162f,#0b1e44)", show_header=True):
    items = []

    if show_header and title.strip():
        header = widgets.HTML(
            f"""
            <div style="
                font-size:14px;
                font-weight:700;
                color:#F8FAFC;
                padding:10px 12px;
                border-bottom:1px solid #243041;
                background:{header_bg};
                box-sizing:border-box;
            ">{title}</div>
            """
        )
        items.append(header)
        body_height = "auto" if height is None else f"calc({height} - 42px)"
    else:
        body_height = "auto" if height is None else height

    body = widgets.VBox(
        [child],
        layout=widgets.Layout(
            padding=padding,
            width="100%",
            height=body_height,
            overflow="hidden"
        )
    )
    items.append(body)

    box_layout = widgets.Layout(
        width=width,
        border='1px solid #243041',
        border_radius='10px',
        background_color='#0B1220',
        margin='2px',
        overflow='hidden'
    )
    if height is not None:
        box_layout.height = height

    return widgets.VBox(items, layout=box_layout)

def build_info_card(label, value, color="#F8FAFC", bg="#0F2747", font_size="20px"):
    return f"""
    <div style="
        padding:10px 12px;
        border-radius:10px;
        border:1px solid rgba(255,255,255,0.06);
        background:{bg};
        box-sizing:border-box;
        width:100%;
        height:100%;
        overflow:hidden;
    ">
        <div style="
            font-size:12px;
            color:#A5B4FC;
            margin-bottom:6px;
            white-space:nowrap;
            overflow:hidden;
            text-overflow:ellipsis;
        ">{label}</div>
        <div style="
            font-size:{font_size};
            font-weight:900;
            color:{color};
            line-height:1.05;
            white-space:nowrap;
            overflow:hidden;
            text-overflow:ellipsis;
        ">{value}</div>
    </div>
    """

# ============================================================
# HTML 模块
# ============================================================

def render_options_analysis_cards(df, spot):
    total_volume = int(df["Volume"].sum()) if "Volume" in df.columns else 0
    call_volume = int(df[df["Type"] == "C"]["Volume"].sum()) if not df.empty else 0
    put_volume = int(df[df["Type"] == "P"]["Volume"].sum()) if not df.empty else 0
    pc_volume_ratio = put_volume / call_volume if call_volume else 0

    total_oi = int(df["OI"].sum()) if "OI" in df.columns else 0
    call_oi = int(df[df["Type"] == "C"]["OI"].sum()) if not df.empty else 0
    put_oi = int(df[df["Type"] == "P"]["OI"].sum()) if not df.empty else 0
    pc_oi_ratio = put_oi / call_oi if call_oi else 0

    try:
        atm_idx = (df["Strike"] - spot).abs().argsort()[:1]
        atm_iv = df.iloc[atm_idx]["IV"].mean() * 100
    except Exception:
        atm_iv = 0

    html = f"""
    <div style="
        display:grid;
        grid-template-columns:repeat(5, minmax(120px, 1fr));
        gap:8px;
        width:100%;
    ">
        {build_info_card("总成交量", format_wan(total_volume), "#F8FAFC", "#12355B", "20px")}
        {build_info_card("P/C 成交比", f"{pc_volume_ratio:.2f}", "#60A5FA", "#12355B", "20px")}
        {build_info_card("总持仓量 (OI)", format_wan(total_oi), "#F8FAFC", "#12355B", "20px")}
        {build_info_card("P/C 持仓比", f"{pc_oi_ratio:.2f}", "#60A5FA", "#12355B", "20px")}
        {build_info_card("ATM IV", f"{atm_iv:.2f}%", "#F472B6", "#12355B", "20px")}
    </div>
    """
    return widgets.HTML(html)

def render_keylevel_cards(max_pain_val, cw, pw, flip, spot):
    flip_str = f"{flip:.2f}" if flip is not None else "--"
    html = f"""
    <div style="
        display:grid;
        grid-template-columns:repeat(5, minmax(115px, 1fr));
        gap:8px;
        width:100%;
    ">
        {build_info_card("Max Pain", f"{max_pain_val:.2f}" if max_pain_val is not None else "--", "#FFF7ED", "#3F6212", "20px")}
        {build_info_card("Call Wall", f"{cw:.2f}" if cw is not None else "--", "#ECFDF5", "#15803D", "20px")}
        {build_info_card("Put Wall", f"{pw:.2f}" if pw is not None else "--", "#FFF7ED", "#C2410C", "20px")}
        {build_info_card("GEX Flip", flip_str, "#FFF7ED", "#B45309", "20px")}
        {build_info_card("Spot", f"{spot:.2f}", "#EFF6FF", "#1D4ED8", "20px")}
    </div>
    """
    return widgets.HTML(html)

# ============================================================
# Gauge + 市场状态
# ============================================================

def render_sentiment_value():
    try:
        temp = ctx.market_temperature(lp.Market.US)
        val = float(getattr(temp, 'temperature', getattr(temp, 'cur', 50)))
        return max(0, min(100, val))
    except Exception:
        return 50.0


def render_sentiment_gauge_html(val):
    return widgets.HTML(f"""
    <div style="
        width:100%;
        display:flex;
        justify-content:center;
        align-items:center;
        padding:8px 0 0 0;
        box-sizing:border-box;
    ">
        <div style="
            position:relative;
            width:320px;
            height:180px;
            overflow:hidden;
        ">
            <!-- 外层彩色半圆 -->
            <div style="
                position:absolute;
                left:0;
                top:18px;
                width:320px;
                height:320px;
                border-radius:50%;
                background:conic-gradient(
                    from 180deg,
                    #EF4444 0deg 54deg,
                    #F59E0B 54deg 126deg,
                    #10B981 126deg 180deg,
                    transparent 180deg 360deg
                );
            "></div>

            <!-- 内层遮罩 -->
            <div style="
                position:absolute;
                left:30px;
                top:48px;
                width:260px;
                height:260px;
                border-radius:50%;
                background:#081426;
            "></div>

            <!-- 标题 -->
            <div style="
                position:absolute;
                top:2px;
                left:0;
                width:100%;
                text-align:center;
                color:#F8FAFC;
                font-size:16px;
                font-weight:700;
            ">市场温度</div>

            <!-- 刻度 -->
            <div style="position:absolute; left:18px; top:86px; color:#F8FAFC; font-size:13px;">20</div>
            <div style="position:absolute; left:84px; top:34px; color:#F8FAFC; font-size:13px;">40</div>
            <div style="position:absolute; left:154px; top:20px; color:#F8FAFC; font-size:13px;">60</div>
            <div style="position:absolute; right:18px; top:86px; color:#F8FAFC; font-size:13px;">80</div>

            <!-- 中间大数字 -->
            <div style="
                position:absolute;
                left:0;
                top:92px;
                width:100%;
                text-align:center;
                color:#E5E7EB;
                font-size:58px;
                font-weight:800;
                line-height:1;
            ">{int(round(val))}</div>
        </div>
    </div>
    """)


def render_sentiment_state_html(val):
    if val is None:
        return widgets.HTML("""
        <div style="
            margin-top:8px;
            padding:10px 12px;
            border-radius:10px;
            background:#081426;
            border:1px solid #1F3554;
            color:#E5E7EB;
            font-size:13px;
            line-height:1.7;
        ">
            暂无市场状态解读。
        </div>
        """)

    if val >= 70:
        state = "偏贪婪"
        hint = "追高谨慎，留意是否回踩关键位后再获得承接。"
        color = "#10B981"
    elif val >= 30:
        state = "中性区间"
        hint = "等待方向确认，重点观察 GEX 与 IV 的配合。"
        color = "#F59E0B"
    else:
        state = "偏恐慌"
        hint = "关注是否出现超跌修复，以及 IV 是否开始回落。"
        color = "#EF4444"

    return widgets.HTML(f"""
    <div style="
        margin-top:8px;
        padding:10px 12px;
        border-radius:10px;
        background:#081426;
        border:1px solid #1F3554;
        color:#E5E7EB;
        font-size:13px;
        line-height:1.7;
    ">
        <div style="font-weight:700;color:#93C5FD;margin-bottom:6px;">市场状态解读</div>
        <div>当前温度：<b style="color:{color};">{val:.0f}</b></div>
        <div>区间判断：<b style="color:{color};">{state}</b></div>
        <div>观察重点：{hint}</div>
    </div>
    """)


def build_sentiment_combo():
    val = render_sentiment_value()
    gauge_html = render_sentiment_gauge_html(val)
    state_html = render_sentiment_state_html(val)

    return widgets.VBox(
        [gauge_html, state_html],
        layout=widgets.Layout(width='100%', height='100%', overflow='hidden')
    )

# ============================================================
# IV 标题
# ============================================================

def build_iv_title(df, spot):
    if df is None or df.empty:
        return "IV Curve"

    call_df = df[df["Type"] == "C"].copy()
    put_df = df[df["Type"] == "P"].copy()

    call_iv_mean = call_df["IV"].mean() * 100 if not call_df.empty else 0
    put_iv_mean = put_df["IV"].mean() * 100 if not put_df.empty else 0

    try:
        atm_idx = (df["Strike"] - spot).abs().argsort()[:1]
        atm_iv = df.iloc[atm_idx]["IV"].mean() * 100
    except Exception:
        atm_iv = 0

    skew = put_iv_mean - call_iv_mean
    if skew > 3:
        skew_text = "Put Skew偏强"
    elif skew < -3:
        skew_text = "Call Skew偏强"
    else:
        skew_text = "Skew中性"

    return (
        f"IV Curve | ATM IV {atm_iv:.2f}% | "
        f"Call {call_iv_mean:.2f}% | Put {put_iv_mean:.2f}% | "
        f"差值 {skew:+.2f}% | {skew_text}"
    )

# ============================================================
# 图表
# ============================================================

def render_k_line(symbol, bars=300):
    try:
        kline_symbol = '.INX' if symbol == 'SPX.US' else symbol
        klines = ctx.candlesticks(kline_symbol, lp.Period.Day, bars, lp.AdjustType.ForwardAdjust)

        rows = []
        for k in klines:
            rows.append({
                "Date": k.timestamp,
                "O": safe_float(k.open),
                "H": safe_float(k.high),
                "L": safe_float(k.low),
                "C": safe_float(k.close),
                "V": int(k.volume or 0)
            })

        df = pd.DataFrame(rows)
        if df.empty:
            return "", None

        df["DateStr"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")
        df = df.sort_values("DateStr").reset_index(drop=True)

        last_row = df.iloc[-1]
        last_c = last_row["C"]
        prev_p = df["C"].iloc[-2] if len(df) > 1 else last_row["O"]
        diff = last_c - prev_p
        pct = diff / prev_p * 100 if prev_p != 0 else 0
        color_theme = "#10B981" if diff >= 0 else "#EF4444"
        vol = int(last_row["V"])

        if vol >= 100000000:
            vol_str = f"{vol/100000000:.2f}亿"
        elif vol >= 10000:
            vol_str = f"{vol/10000:.2f}万"
        else:
            vol_str = str(vol)

        status_html = f"""
        <div style="
            font-family:Arial;padding:10px 12px;background:#081426;border-radius:10px;
            border:1px solid #1F3554;margin-bottom:8px;color:#F8FAFC;
        ">
            <span style="font-size:22px;font-weight:900;color:#60A5FA;">{symbol}</span>
            <span style="margin-left:10px;font-size:12px;background:#1E293B;padding:3px 8px;border-radius:4px;color:#CBD5E1;">{last_row['DateStr']}</span>
            <span style="margin-left:14px;font-size:13px;">现价 <b style="color:{color_theme};">{last_c:.2f}</b></span>
            <span style="margin-left:10px;font-size:13px;">涨跌 <b style="color:{color_theme};">{diff:+.2f} ({pct:+.2f}%)</b></span>
            <span style="margin-left:10px;font-size:13px;">成交量 <b style="color:#93C5FD;">{vol_str}</b></span>
        </div>
        """

        fig = go.Figure(
            make_subplots(
                rows=2, cols=1,
                shared_xaxes=True,
                vertical_spacing=0.02,
                row_heights=[0.78, 0.22]
            )
        )

        fig.add_trace(
            go.Candlestick(
                x=df["DateStr"],
                open=df["O"], high=df["H"], low=df["L"], close=df["C"],
                increasing_line_color="#10B981",
                decreasing_line_color="#EF4444",
                showlegend=False
            ),
            row=1, col=1
        )

        bar_colors = ['#047857' if c >= o else '#991B1B' for o, c in zip(df["O"], df["C"])]
        fig.add_trace(
            go.Bar(
                x=df["DateStr"], y=df["V"],
                marker_color=bar_colors,
                showlegend=False
            ),
            row=2, col=1
        )

        fig.update_layout(
            height=410,
            template='plotly_dark',
            paper_bgcolor='#081426',
            plot_bgcolor='#081426',
            xaxis_rangeslider_visible=False,
            margin=dict(t=2, b=2, l=8, r=8),
            font=dict(color="#E5E7EB"),
            autosize=True
        )
        fig.update_xaxes(showgrid=True, gridcolor="#1F2937")
        fig.update_yaxes(showgrid=True, gridcolor="#1F2937")
        return status_html, fig
    except Exception as e:
        return f"<b style='color:#F87171'>K线加载失败: {e}</b>", None

# ============================================================
# 控件
# ============================================================

SYMBOL_LIST = ['SPY.US', 'QQQ.US', 'AMZN.US', 'META.US', 'MU.US', 'TSLA.US', 'NVDA.US', 'AAPL.US', 'GOOG.US']

symbol_sel = widgets.Dropdown(options=SYMBOL_LIST, value='SPY.US', description='Ticker:', layout={'width': '220px'})
date_sel = widgets.Dropdown(options=[], description='Expiry:', layout={'width': '210px'})
win_s = widgets.IntSlider(value=40, min=10, max=150, description="Window", layout={'width': '240px'})
rate_s = widgets.FloatSlider(value=0.037, min=0, max=0.1, step=0.0001, description="Rate", readout_format='.4f', layout={'width': '240px'})
div_s = widgets.FloatSlider(value=0.012, min=0, max=0.05, step=0.0001, description="Div", readout_format='.4f', layout={'width': '230px'})
run_btn = widgets.Button(description="SYNC DASHBOARD", button_style="success", icon="bolt", layout={'width': '190px'})

# 输出区
k_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
sentiment_combo_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
options_summary_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
keylevels_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
gex_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
iv_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
agg_table_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
detail_table_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))
status_out = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))

current_iv_title = "IV Curve"

# ============================================================
# 逻辑
# ============================================================

def on_symbol_change(change):
    ticker = symbol_sel.value

    try:
        all_dates = ctx.option_chain_expiry_date_list(ticker)
        valid_dates = [d for d in all_dates if d >= dt.date.today()]
        date_sel.options = [(str(d), d) for d in valid_dates[:20]]
        if valid_dates:
            date_sel.value = valid_dates[0]
    except Exception:
        date_sel.options = []

    with k_out:
        clear_output(wait=True)
        status_html, fig = render_k_line(ticker, bars=300)
        if status_html:
            display(widgets.HTML(value=status_html))
        if fig is not None:
            display(fig)

    with sentiment_combo_out:
        clear_output(wait=True)
        display(build_sentiment_combo())

def on_sync_click(_):
    global current_iv_title
    ticker = symbol_sel.value
    clear_outputs(
        options_summary_out, keylevels_out,
        gex_out, iv_out,
        agg_table_out, detail_table_out, status_out
    )

    with status_out:
        print(f"🚀 正在分析 {ticker} 深度数据...")

    try:
        quote_symbol = '.INX' if ticker == 'SPX.US' else ticker
        quote_res = ctx.quote([quote_symbol])
        if not quote_res:
            with status_out:
                clear_output(wait=True)
                print("⚠️ 行情获取失败。")
            return

        spot = float(quote_res[0].last_done)
        selected_expiry = date_sel.value
        if selected_expiry is None:
            with status_out:
                clear_output(wait=True)
                print("⚠️ 请先选择到期日。")
            return

        all_rows = []
        sym_map = {}

        chain = ctx.option_chain_info_by_date(ticker, selected_expiry)
        for item in (chain or []):
            try:
                strike = float(item.price)
            except Exception:
                continue

            if abs(strike - spot) <= win_s.value:
                sym_map[item.call_symbol] = {"K": strike, "T": "C", "Exp": selected_expiry}
                sym_map[item.put_symbol] = {"K": strike, "T": "P", "Exp": selected_expiry}

        if not sym_map:
            with status_out:
                clear_output(wait=True)
                print("⚠️ 未抓取到期权。请尝试调大 Window。")
            return

        symbols = list(sym_map.keys())

        calc_indexes = [lp.CalcIndex.Delta, lp.CalcIndex.Gamma, lp.CalcIndex.ImpliedVolatility]
        if hasattr(lp.CalcIndex, "Theta"):
            calc_indexes.append(lp.CalcIndex.Theta)
        if hasattr(lp.CalcIndex, "Vega"):
            calc_indexes.append(lp.CalcIndex.Vega)

        for i in range(0, len(symbols), 100):
            batch = symbols[i:i + 100]
            calc_res = ctx.calc_indexes(batch, calc_indexes)
            quotes = ctx.option_quote(batch)

            oi_dict = {q.symbol: int(q.open_interest or 0) for q in quotes}
            vol_dict = {q.symbol: int(q.volume or 0) for q in quotes}

            for res in calc_res:
                if res.symbol not in sym_map:
                    continue

                m = sym_map[res.symbol]
                oi = oi_dict.get(res.symbol, 0)
                gamma = safe_float(getattr(res, "gamma", 0.0), 0.0)
                delta = safe_float(getattr(res, "delta", 0.0), 0.0)
                theta = safe_float(getattr(res, "theta", 0.0), 0.0)
                vega = safe_float(getattr(res, "vega", 0.0), 0.0)
                iv = safe_float(getattr(res, "implied_volatility", 0.0), 0.0)
                iv = iv / 100.0 if iv > 1.5 else iv

                all_rows.append({
                    "Strike": m["K"],
                    "Type": m["T"],
                    "Expiry": m["Exp"],
                    "OI": oi,
                    "Volume": vol_dict.get(res.symbol, 0),
                    "IV": iv,
                    "Delta": delta,
                    "Gamma": gamma,
                    "Theta": theta,
                    "Vega": vega,
                    "RawGEX": (gamma * oi * 100 * (spot ** 2) * 0.01) / 1e6
                })

        df = pd.DataFrame(all_rows)
        if df.empty:
            with status_out:
                clear_output(wait=True)
                print("⚠️ 期权结果为空。")
            return

        current_iv_title = build_iv_title(df, spot)

        max_pain_val, strikes_list, pains_list = calculate_max_pain(df)

        agg = df.groupby("Strike").apply(
            lambda x: pd.Series({
                "CallGEX": x[x["Type"] == "C"]["RawGEX"].sum(),
                "PutGEX": x[x["Type"] == "P"]["RawGEX"].sum(),
                "NetGEX": x[x["Type"] == "C"]["RawGEX"].sum() - x[x["Type"] == "P"]["RawGEX"].sum(),
                "TotalOI": x["OI"].sum(),
                "TotalVolume": x["Volume"].sum()
            }),
            include_groups=False
        ).reset_index()

        agg = pd.merge(
            agg,
            pd.DataFrame({"Strike": strikes_list, "TotalPain": pains_list}),
            on="Strike",
            how="left"
        )

        flip = interp_flip(agg)
        cw = agg.loc[agg["CallGEX"].idxmax(), "Strike"] if not agg.empty else None
        pw = agg.loc[agg["PutGEX"].idxmax(), "Strike"] if not agg.empty else None

        with options_summary_out:
            clear_output(wait=True)
            display(render_options_analysis_cards(df, spot))

        with keylevels_out:
            clear_output(wait=True)
            display(render_keylevel_cards(max_pain_val, cw, pw, flip, spot))

        # ---------- GEX ----------
        max_abs_gex = max(float(agg["NetGEX"].abs().max()), 1.0)
        fig_gex = go.Figure(
            go.Bar(
                y=agg["Strike"],
                x=agg["NetGEX"],
                orientation="h",
                marker_color=["#10B981" if x >= 0 else "#EF4444" for x in agg["NetGEX"]],
                name="Net GEX"
            )
        )

        line_specs = [
            (spot, "Spot", "#3B82F6", "solid"),
            (cw, "Call Wall", "#10B981", "dash"),
            (pw, "Put Wall", "#EF4444", "dash"),
            (max_pain_val, "Max Pain", "#F59E0B", "dot"),
            (flip, "Flip", "#8B5CF6", "dashdot"),
        ]

        for v, n, c, d in line_specs:
            if v is not None:
                fig_gex.add_hline(
                    y=v,
                    line=dict(color=c, width=1.6, dash=d),
                    annotation_text=f"{n}: {v:.2f}",
                    annotation_position="top right"
                )

        fig_gex.update_layout(
            title="GEX Profile",
            template="plotly_dark",
            paper_bgcolor="#081426",
            plot_bgcolor="#081426",
            height=430,
            margin=dict(t=40, b=20, l=20, r=20),
            xaxis=dict(range=[-max_abs_gex * 1.15, max_abs_gex * 1.15], title="GEX / Gamma"),
            yaxis=dict(title="Strike"),
            font=dict(color="#E5E7EB"),
            autosize=True
        )
        fig_gex.update_xaxes(showgrid=True, gridcolor="#1F2937")
        fig_gex.update_yaxes(showgrid=True, gridcolor="#1F2937")

        with gex_out:
            clear_output(wait=True)
            display(fig_gex)

        # ---------- IV ----------
        fig_iv = go.Figure()
        c_iv = df[df["Type"] == "C"].groupby("Strike")["IV"].mean()
        p_iv = df[df["Type"] == "P"].groupby("Strike")["IV"].mean()

        fig_iv.add_trace(go.Scatter(
            x=c_iv.index, y=c_iv.values * 100,
            name="Call IV", line=dict(color="#10B981")
        ))
        fig_iv.add_trace(go.Scatter(
            x=p_iv.index, y=p_iv.values * 100,
            name="Put IV", line=dict(color="#EF4444")
        ))

        fig_iv.add_vline(
            x=spot,
            line=dict(color="#3B82F6", width=1.5, dash="dash"),
            annotation_text=f"Spot: {spot:.2f}",
            annotation_position="top right"
        )

        fig_iv.update_layout(
            title="IV Curve (Volatility Smile)",
            template="plotly_dark",
            paper_bgcolor="#081426",
            plot_bgcolor="#081426",
            height=430,
            margin=dict(t=40, b=20, l=20, r=20),
            xaxis_title="Strike",
            yaxis_title="IV %",
            font=dict(color="#E5E7EB"),
            autosize=True
        )
        fig_iv.update_xaxes(showgrid=True, gridcolor="#1F2937")
        fig_iv.update_yaxes(showgrid=True, gridcolor="#1F2937")

        with iv_out:
            clear_output(wait=True)
            display(fig_iv)

        # ---------- 表格 ----------
        agg_show = agg[["Strike", "CallGEX", "PutGEX", "NetGEX", "TotalOI", "TotalVolume", "TotalPain"]].round(4)
        df_show = df[["Strike", "Type", "Expiry", "OI", "Volume", "IV", "Delta", "Gamma", "Theta", "Vega", "RawGEX"]].round(6)

        with agg_table_out:
            clear_output(wait=True)
            display(create_pagination_widget(agg_show, rows_per_page=10))

        with detail_table_out:
            clear_output(wait=True)
            display(create_pagination_widget(df_show, rows_per_page=10))

        with status_out:
            clear_output(wait=True)
            print("✅ 结构化分析同步完成！")

        rebuild_quant_section()

    except Exception:
        import traceback
        with status_out:
            clear_output(wait=True)
            traceback.print_exc()

# ============================================================
# 登录
# ============================================================

def handle_login(b):
    global ctx
    btn_login.description = "正在验证..."
    try:
        ctx = lp.QuoteContext(
            lp.Config(
                app_key=in_app_key.value,
                app_secret=in_app_sec.value,
                access_token=in_token.value
            )
        )
        ctx.market_temperature(lp.Market.US)

        login_ui.layout.display = 'none'
        terminal_ui.layout.display = 'block'
        on_symbol_change(None)
        on_sync_click(None)
        btn_login.description = "Connect Server"
    except Exception:
        btn_login.description = "连接失败，请检查 Token"

# ============================================================
# 动态重建 Quant 区域
# ============================================================

quant_container = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))

def rebuild_quant_section():
    with quant_container:
        clear_output(wait=True)

        quant_left = widgets.VBox(
            [card_box(gex_out, "GEX Profile", width="100%", height="470px")],
            layout=widgets.Layout(width='50%', min_width='600px', overflow='hidden')
        )

        quant_right = widgets.VBox(
            [card_box(iv_out, current_iv_title, width="100%", height="470px", padding="6px")],
            layout=widgets.Layout(width='50%', min_width='600px', overflow='hidden')
        )

        quant_analysis = widgets.HBox(
            [quant_left, quant_right],
            layout=widgets.Layout(width='100%', flex_flow='row wrap', align_items='flex-start', overflow='hidden')
        )

        display(quant_analysis)

# ============================================================
# 绑定事件
# ============================================================

symbol_sel.observe(on_symbol_change, names='value')
run_btn.on_click(on_sync_click)

# ============================================================
# 布局
# ============================================================

login_ui = widgets.VBox(
    [
        widgets.HTML("<h2 style='color:#60A5FA;'>🔒 量化策略终端接入</h2>"),
        (in_app_key := widgets.Password(description="App Key:")),
        (in_app_sec := widgets.Password(description="Secret:")),
        (in_token := widgets.Password(description="Token:")),
        (btn_login := widgets.Button(description="Connect Server", button_style="info", layout={'width': '300px', 'margin': '20px 0'}))
    ],
    layout=widgets.Layout(align_items='center', margin='80px')
)
btn_login.on_click(handle_login)

header_bar = widgets.HBox(
    [symbol_sel, date_sel, win_s, rate_s, div_s, run_btn],
    layout=widgets.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='space-between',
        padding='10px 12px',
        border='1px solid #243041',
        border_radius='10px',
        background_color='#1F2937',
        margin='4px 2px 8px 2px',
        gap='8px',
        overflow='hidden'
    )
)

# 第二级：尽量稳，不再过度复杂
macro_left_k = widgets.VBox(
    [card_box(k_out, title="", width="100%", height="470px", padding="6px", show_header=False)],
    layout=widgets.Layout(width='38%', min_width='520px', overflow='hidden')
)

gauge_col = widgets.VBox(
    [card_box(sentiment_combo_out, "Market Temperature Gauge", width="100%", height="470px", padding="6px")],
    layout=widgets.Layout(width='24%', min_width='380px', overflow='hidden')
)

summary_right = widgets.VBox(
    [
        card_box(
            options_summary_out,
            "Options Analysis",
            width="100%",
            height="210px",
            header_bg="linear-gradient(90deg,#12355B,#0B1F3A)"
        ),
        card_box(
            keylevels_out,
            "Key Levels",
            width="100%",
            height="256px",
            header_bg="linear-gradient(90deg,#7C2D12,#B45309)"
        )
    ],
    layout=widgets.Layout(width='38%', min_width='660px', overflow='hidden')
)

market_dashboard = widgets.HBox(
    [macro_left_k, gauge_col, summary_right],
    layout=widgets.Layout(
        width='100%',
        flex_flow='row nowrap',
        align_items='flex-start',
        justify_content='space-between',
        overflow='hidden'
    )
)

rebuild_quant_section()

# 底部表格 50/50 并排，避免消失
raw_left = widgets.VBox(
    [card_box(agg_table_out, "Strike Aggregated GEX / Gamma", width="100%")],
    layout=widgets.Layout(width='50%', overflow='hidden')
)

raw_right = widgets.VBox(
    [card_box(detail_table_out, "Option Details / Greeks", width="100%")],
    layout=widgets.Layout(width='50%', overflow='hidden')
)

raw_data = widgets.HBox(
    [raw_left, raw_right],
    layout=widgets.Layout(
        width='100%',
        flex_flow='row nowrap',
        align_items='stretch',
        justify_content='space-between',
        overflow='hidden'
    )
)

terminal_ui = widgets.VBox(
    [
        widgets.HTML("""
        <div style="
            text-align:center;
            font-size:30px;
            font-weight:900;
            letter-spacing:1px;
            color:#F8FAFC;
            padding:14px 0 10px 0;
        ">ULTIMATE STRATEGY TERMINAL</div>
        """),
        header_bar,
        market_dashboard,
        quant_container,
        card_box(status_out, "System Status", width="100%"),
        raw_data
    ],
    layout=widgets.Layout(
        width='100%',
        padding='10px',
        background_color='#20242B',
        overflow='hidden'
    )
)
terminal_ui.layout.display = 'none'

display(widgets.VBox([login_ui, terminal_ui], layout=widgets.Layout(width='100%', overflow='hidden')))